# CS-781 Phase 7 - Prior Case Retrieval with our Phase-1 legal LM
**Settings:** Accelerator **GPU T4 x2**, Internet **On**, secret **HF_TOKEN** switched ON. Run the cells top to bottom (about 3 hours).
Everything is saved to `<you>/cs781-p1-ckpt/phase7/`; finished runs are skipped if you run again.

1. Cell 1 - login, our tokenizer. 2. Cell 3 - downloads `Phase_7_PCR`, cleans + tokenises every case once, builds BM25 (words and
word pairs, settings chosen on TRAIN), TF-IDF, the latest year in each case, BM25 hard negatives; cached on HF.
3. Cell 7 - safety test on the real data, then 7 runs on both GPUs: our Phase-1 llama fine-tuned as a retriever with
**LoRMA** (compulsory), **full fine-tuning**, **partial (top-4-layer) fine-tuning**, **IA3**; **knowledge distillation** llama -> 2x smaller
shared model (and the same student without KD); the Phase-1 model with no fine-tuning (transfer check).
Each case = windows of 512 tokens spread over the whole case; case vector = mean of the window vectors, or MaxSim over windows;
contrastive training with in-batch and BM25 hard negatives; best epoch chosen on DEV.
4. Cell 8 - every run scored with the OFFICIAL `evaluate_phase7.py` on DEV (retriever alone and fused with BM25 + year rule + hub +
feedback, weights chosen on dev with a 2-fold honesty check), ensembles of several retrievers, all zips validated against the test
pool, and the **SUBMISSION ORDER**.

In [1]:
# ===================== P7 CELL 1 - Setup (Phase 7: Prior Case Retrieval) =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF_TOKEN" (or "HF") switched ON for this notebook.
import os, sys, json, time, math, random, re, shutil, collections, subprocess, threading
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = _sec.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-7 outputs go to its phase7/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P7        = "/tmp/cs781/p7"; os.makedirs(P7, exist_ok=True)
RAW       = f"{P7}/flat"; os.makedirs(RAW, exist_ok=True)
DATA      = f"{P7}/data"; os.makedirs(DATA, exist_ok=True)
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P7}/tok")
shutil.copy(TOK_PATH, f"{DATA}/tokenizer.json")
import torch, sklearn, scipy
print(f"HF user {HF_USER} | torch {torch.__version__} | sklearn {sklearn.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


tokenizer.json:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

HF user Abhayk24 | torch 2.11.0+cu130 | sklearn 1.6.1 | GPUs 2 Tesla T4 Tesla T4


In [2]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Writing /kaggle/working/model.py


In [3]:
%%writefile /kaggle/working/p7data.py
"""CS-781 Phase 7 (Prior Case Retrieval): read every split once, clean the text like Phase 1, tokenise it with OUR tokenizer,
and build the lexical (no-model) signals used next to our Phase-1 retrievers:
  - BM25 over words and over word pairs (k1, b chosen on the TRAIN split with its labels),
  - TF-IDF vectors of the candidates (candidate-candidate similarity for pseudo-relevance feedback),
  - the latest year mentioned in each case (a PRIOR case cannot be later than the query),
  - BM25 hard negatives for every training query.
Outputs -> <out_dir>/{split}_tok.npz, lex_{split}.npz, tfidf_{split}.npz, labels_{split}.json, {split}_candidate_ids.json, stats.json
"""
import argparse, json, os, re, time, unicodedata
import numpy as np
import scipy.sparse as sp

PUNCT = {"‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'", "“": '"', "”": '"', "„": '"',
         "″": '"', "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-", "―": "-", "−": "-",
         "•": "*", "·": "."}
_TABLE = str.maketrans(PUNCT)
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS = re.compile(r"\s+")
_YEAR = re.compile(r"(?<!\d)(1[89]\d\d|20[0-2]\d)(?!\d)")
K = 7


def normalize(t):
    """Same cleaning as Phases 1-6 (fancy punctuation -> ASCII, accents stripped, control chars dropped, spaces collapsed)."""
    if isinstance(t, list):
        t = " ".join(str(x) for x in t if x is not None)
    elif not isinstance(t, str):
        t = str(t or "")
    t = t.translate(_TABLE)
    if not t.isascii():
        t = unicodedata.normalize("NFKC", t)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return _WS.sub(" ", t).strip()


def _text_of(v):
    if isinstance(v, dict):
        for k in ("text", "document", "sentences", "content", "judgment"):
            if k in v:
                return v[k]
        return ""
    return v


def load_queries(path):
    """-> {qid: (raw text, [relevant ids] or None)}; accepts {qid: {...}} or [{id, text, relevant_candidates}]."""
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, list):
        d = {str(r.get("id", r.get("query_id", r.get("qid")))): r for r in d}
    out = {}
    for q, v in d.items():
        rel = v.get("relevant_candidates") if isinstance(v, dict) else None
        if rel is not None:
            rel = [str(x) for x in rel if isinstance(x, str) and x.strip()]
        out[str(q)] = (_text_of(v), rel)
    return out


def load_candidates(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, list):
        d = {str(r.get("id", r.get("candidate_id", r.get("cid")))): r for r in d}
    return {str(c): _text_of(v) for c, v in d.items()}


def latest_year(text):
    ys = [int(y) for y in _YEAR.findall(text)]
    ys = [y for y in ys if y <= 2026]
    return max(ys) if ys else 0


def bm25_matrix(tf, k1, b):
    """tf: (C x V) csr counts -> BM25-saturated, idf-weighted document matrix."""
    tf = tf.tocsr().astype(np.float32)
    N = tf.shape[0]
    dl = np.asarray(tf.sum(1)).ravel()
    avg = max(dl.mean(), 1.0)
    df = np.bincount(tf.indices, minlength=tf.shape[1]).astype(np.float32)
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(tf.indptr))
    x = tf.data
    sat = x * (k1 + 1) / (x + k1 * (1 - b + b * dl[rows] / avg))
    W = sp.csr_matrix((sat * idf[tf.indices], tf.indices, tf.indptr), shape=tf.shape)
    return W


def bm25_scores(qtf, W):
    qb = qtf.tocsr().copy().astype(np.float32); qb.data[:] = 1.0          # each query term once (queries are whole cases)
    return np.asarray((qb @ W.T).todense(), dtype=np.float32)


def self_mask(qids, cids):
    ci = {c: j for j, c in enumerate(cids)}
    return [(i, ci[q]) for i, q in enumerate(qids) if q in ci]


def topk_ids(S, cids, k=K):
    idx = np.argpartition(-S, kth=min(k, S.shape[1] - 1), axis=1)[:, :k]
    out = []
    for i in range(S.shape[0]):
        r = idx[i][np.argsort(-S[i, idx[i]], kind="stable")]
        out.append([cids[j] for j in r])
    return out


def micro_f1(pred, gold):
    """pred: {qid: [7 ids]}, gold: {qid: [ids]} -> (f1, precision, recall) exactly as evaluate_phase7.py."""
    tp = fp = fn = 0
    for q, g in gold.items():
        g = set(g); p = set(pred[q][:K]); tp += len(g & p); fp += len(p - g); fn += len(g - p)
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    return f1, (tp / (tp + fp) if tp + fp else 0.0), (tp / (tp + fn) if tp + fn else 0.0)


def f1_of_scores(S, qids, cids, gold):
    p = dict(zip(qids, topk_ids(S, cids)))
    return micro_f1(p, {q: gold[q] for q in qids})[0]


def zrow(S):
    S = S.astype(np.float32, copy=True)
    fin = np.isfinite(S)
    Sm = np.where(fin, S, np.nan)
    mu = np.nanmean(Sm, 1, keepdims=True); sd = np.nanstd(Sm, 1, keepdims=True) + 1e-6
    Z = (S - mu) / sd
    Z[~fin] = -1e4
    return Z


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw_dir", required=True)
    ap.add_argument("--tok", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--n_hard", type=int, default=50)
    a = ap.parse_args(argv)
    os.makedirs(a.out_dir, exist_ok=True)
    from tokenizers import Tokenizer
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    tok = Tokenizer.from_file(a.tok)
    stats = {}
    splits = {}
    for s in ["train", "dev", "test"]:
        t0 = time.time()
        Q = load_queries(os.path.join(a.raw_dir, f"{s}.json"))
        C = load_candidates(os.path.join(a.raw_dir, f"{s}_candidates.json"))
        qids, cids = list(Q), list(C)
        qtxt = [normalize(Q[q][0]) for q in qids]; ctxt = [normalize(C[c]) for c in cids]
        has = any(Q[q][1] is not None for q in qids)
        labels = {q: (Q[q][1] or []) for q in qids} if has else {}
        json.dump(cids, open(os.path.join(a.out_dir, f"{s}_candidate_ids.json"), "w"))
        if labels:
            json.dump(labels, open(os.path.join(a.out_dir, f"labels_{s}.json"), "w"))
        # ---- our tokenizer ----
        def pack(txts):
            enc = tok.encode_batch(txts, add_special_tokens=False)
            lens = [len(e.ids) for e in enc]
            arr = np.fromiter((i for e in enc for i in e.ids), dtype=np.uint16, count=sum(lens))
            return arr, np.concatenate([[0], np.cumsum(lens)]).astype(np.int64), lens
        qt, qo, ql = pack(qtxt); ct, co, cl = pack(ctxt)
        qy = np.array([latest_year(t) for t in qtxt], np.int16); cy = np.array([latest_year(t) for t in ctxt], np.int16)
        np.savez(os.path.join(a.out_dir, f"{s}_tok.npz"), q_tok=qt, q_off=qo, c_tok=ct, c_off=co,
                 q_ids=np.array(qids), c_ids=np.array(cids), q_year=qy, c_year=cy)
        rel_in_pool = [sum(r in set(cids) for r in labels.get(q, [])) for q in qids] if labels else []
        stats[s] = {"queries": len(qids), "candidates": len(cids), "query_tokens_mean": float(np.mean(ql)),
                    "query_tokens_median": float(np.median(ql)), "cand_tokens_mean": float(np.mean(cl)),
                    "cand_tokens_median": float(np.median(cl)), "queries_inside_pool": sum(q in set(cids) for q in qids),
                    "year_found_queries": float(np.mean(qy > 0)), "year_found_cands": float(np.mean(cy > 0))}
        if labels:
            nrel = [len(v) for v in labels.values()]
            stats[s].update(relevant_mean=float(np.mean(nrel)), relevant_median=float(np.median(nrel)),
                            relevant_max=int(np.max(nrel)), relevant_in_pool_share=float(np.sum(rel_in_pool) / max(1, np.sum(nrel))),
                            queries_without_relevant=int(sum(n == 0 for n in nrel)))
        # ---- lexical vectors (fit on this split's candidate pool) ----
        cv1 = CountVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", dtype=np.float32, min_df=1)
        C1 = cv1.fit_transform(ctxt); Q1 = cv1.transform(qtxt)
        cv2 = CountVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", ngram_range=(2, 2),
                              min_df=2, max_df=0.5, dtype=np.float32)
        C2 = cv2.fit_transform(ctxt); Q2 = cv2.transform(qtxt)
        tfv = TfidfVectorizer(lowercase=True, stop_words="english", sublinear_tf=True, min_df=2, max_df=0.5, dtype=np.float32)
        T = tfv.fit_transform(ctxt)
        sp.save_npz(os.path.join(a.out_dir, f"tfidf_{s}.npz"), T.tocsr())
        splits[s] = dict(qids=qids, cids=cids, C1=C1, Q1=Q1, C2=C2, Q2=Q2, labels=labels, qy=qy, cy=cy)
        print(f"[data] {s}: {len(qids)} queries, {len(cids)} candidates | vocab {len(cv1.vocabulary_)} words, {len(cv2.vocabulary_)} pairs"
              f" | {time.time() - t0:.0f}s", flush=True)

    # ---- BM25 k1/b chosen on TRAIN labels (dev stays clean for model selection) ----
    tr = splits["train"]
    ms = self_mask(tr["qids"], tr["cids"])
    best = {}
    for name, Cm, Qm in [("bm25u", tr["C1"], tr["Q1"]), ("bm25b", tr["C2"], tr["Q2"])]:
        res = []
        for k1 in [0.9, 1.5, 2.5]:
            for b in [0.5, 0.75, 1.0]:
                S = bm25_scores(Qm, bm25_matrix(Cm, k1, b))
                for i, j in ms: S[i, j] = -np.inf
                res.append((f1_of_scores(S, tr["qids"], tr["cids"], tr["labels"]), k1, b))
        res.sort(reverse=True)
        best[name] = {"k1": res[0][1], "b": res[0][2], "train_f1": res[0][0]}
        print(f"[data] {name}: best on train k1={res[0][1]} b={res[0][2]} F1@7={res[0][0]:.4f}", flush=True)
    stats["bm25"] = best
    for s, d in splits.items():
        Su = bm25_scores(d["Q1"], bm25_matrix(d["C1"], best["bm25u"]["k1"], best["bm25u"]["b"]))
        Sb = bm25_scores(d["Q2"], bm25_matrix(d["C2"], best["bm25b"]["k1"], best["bm25b"]["b"]))
        for i, j in self_mask(d["qids"], d["cids"]):
            Su[i, j] = -np.inf; Sb[i, j] = -np.inf
        np.savez(os.path.join(a.out_dir, f"lex_{s}.npz"), bm25u=Su, bm25b=Sb, q_ids=np.array(d["qids"]), c_ids=np.array(d["cids"]),
                 q_year=d["qy"], c_year=d["cy"])
        if d["labels"]:
            for nm, S in [("bm25u", Su), ("bm25b", Sb), ("bm25u+b", zrow(Su) + zrow(Sb))]:
                stats[s][f"F1@7_{nm}"] = f1_of_scores(S, d["qids"], d["cids"], d["labels"])
            print(f"[data] {s} lexical F1@7: " + " | ".join(f"{k[5:]} {v:.4f}" for k, v in stats[s].items() if k.startswith("F1@7")), flush=True)
        if s == "train":                                          # BM25 hard negatives for every training query
            Z = zrow(Su) + zrow(Sb); hard = {}
            for i, q in enumerate(d["qids"]):
                rel = set(d["labels"].get(q, [])); order = np.argsort(-Z[i])[: a.n_hard + len(rel) + 2]
                hard[q] = [d["cids"][j] for j in order if d["cids"][j] not in rel and d["cids"][j] != q][: a.n_hard]
            json.dump(hard, open(os.path.join(a.out_dir, "hard_train.json"), "w"))
    json.dump(stats, open(os.path.join(a.out_dir, "stats.json"), "w"), indent=1)
    print("[data] done", flush=True)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p7data.py


In [4]:
%%writefile /kaggle/working/p7fuse.py
"""CS-781 Phase 7: combine signals into ONE ranking per query (no outside model).
score(q, c) = sum_i w_i * z_i(q, c)    (every signal z-normalised within the query's candidate pool)
signals: bm25u, bm25b (lexical) | dense_mean, dense_max of one or more of OUR Phase-1 retrievers | year (candidate later than
the query: cannot be a PRIOR case) | hub (how often a candidate is a strong match across all queries of the split)
then pseudo-relevance feedback: + w_prf * z(mean TF-IDF similarity of c to the query's current top-10).
The few weights are chosen on DEV by coordinate ascent (model selection); a 2-fold split of dev checks they generalise.
Also: `python p7fuse.py ...` = the ENSEMBLE of several finished runs.
"""
import argparse, json, os, shutil, zipfile
import numpy as np
import scipy.sparse as sp

from p7data import zrow, topk_ids, micro_f1, K

GRID = {"pos": [0.0, 0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0], "year": [0.0, -0.5, -1.0, -2.0, -4.0], "hub": [-1.0, -0.5, 0.0, 0.5, 1.0],
        "prf": [0.0, 0.25, 0.5, 1.0, 2.0]}


class Split:
    def __init__(self, data_dir, split, dense=None):
        z = np.load(os.path.join(data_dir, f"lex_{split}.npz"))
        self.qids = [str(x) for x in z["q_ids"]]; self.cids = [str(x) for x in z["c_ids"]]
        self.f = {"bm25u": zrow(z["bm25u"]), "bm25b": zrow(z["bm25b"])}
        self.invalid = ~np.isfinite(z["bm25u"])                                  # the query itself
        qy, cy = z["q_year"].astype(np.int32), z["c_year"].astype(np.int32)
        self.f["year"] = ((cy[None, :] > qy[:, None]) & (qy[:, None] > 0) & (cy[None, :] > 0)).astype(np.float32)
        lab = os.path.join(data_dir, f"labels_{split}.json")
        self.gold = json.load(open(lab)) if os.path.exists(lab) else None
        self.T = sp.load_npz(os.path.join(data_dir, f"tfidf_{split}.npz")).tocsr()
        self._cc = None
        for name, S in (dense or {}).items():
            self.add(name, S)

    def add(self, name, S):
        S = S.astype(np.float32).copy(); S[self.invalid] = -np.inf
        self.f[name] = zrow(S)

    def cc(self):
        if self._cc is None:
            self._cc = np.asarray((self.T @ self.T.T).todense(), dtype=np.float32) if self.T.shape[0] <= 12000 else False
        return self._cc


def base_score(sp_, w):
    S = np.zeros_like(sp_.f["bm25u"])
    for k, v in w.items():
        if v and k in sp_.f:
            S += v * sp_.f[k]
    if w.get("hub"):
        lex = sp_.f["bm25u"] + sp_.f["bm25b"]
        h = np.where(lex > -1e3, lex, np.nan); h = np.nanmean(h, 0)
        S += w["hub"] * ((h - h.mean()) / (h.std() + 1e-6))[None, :]
    S[sp_.invalid] = -1e9
    return S


def score(sp_, w):
    S = base_score(sp_, w)
    if w.get("prf"):
        cc = sp_.cc()
        if cc is not False:
            top = np.argpartition(-S, 10, axis=1)[:, :10]
            P = cc[top].mean(1)                                  # (Q, C): mean similarity to the current top-10
            P[sp_.invalid] = np.nan
            mu = np.nanmean(P, 1, keepdims=True); sd = np.nanstd(P, 1, keepdims=True) + 1e-6
            S = S + w["prf"] * np.nan_to_num((P - mu) / sd, nan=0.0)
            S[sp_.invalid] = -1e9
    return S


def f1(sp_, w, rows=None):
    S = score(sp_, w)
    qids = sp_.qids if rows is None else [sp_.qids[i] for i in rows]
    if rows is not None:
        S = S[rows]
    pred = dict(zip(qids, topk_ids(S, sp_.cids)))
    return micro_f1(pred, {q: sp_.gold[q] for q in qids})[0]


def tune(sp_, names, rows=None, passes=3, require=()):
    """coordinate ascent over the weights of `names` (+ year, hub, prf); starts from the best single signal.
    require: at least one of these signals keeps a non-zero weight (a run's submission always uses its own retriever)."""
    singles = {n: f1(sp_, {n: 1.0}, rows) for n in names}
    w = {n: 0.0 for n in names}; w[max(singles, key=singles.get)] = 1.0
    if require and not any(w[r] for r in require):
        w[max(require, key=lambda r: singles[r])] = 0.5
    w.update(year=0.0, hub=0.0, prf=0.0)
    best = f1(sp_, w, rows)
    for _ in range(passes):
        improved = False
        for k in list(w):
            grid = GRID[k] if k in GRID else GRID["pos"]
            for v in grid:
                if v == w[k]:
                    continue
                t = dict(w); t[k] = v
                if not any(t[n] for n in names) or (require and not any(t[r] for r in require)):
                    continue
                s = f1(sp_, t, rows)
                if s > best + 1e-9:
                    best, w, improved = s, t, True
        if not improved:
            break
    return w, best, singles


def tune_report(sp_, names, require=()):
    w, best, singles = tune(sp_, names, require=require)
    n = len(sp_.qids); rng = np.random.RandomState(0); perm = rng.permutation(n); A, B = perm[: n // 2], perm[n // 2:]
    wa, _, _ = tune(sp_, names, A, require=require); wb, _, _ = tune(sp_, names, B, require=require)
    cross = (f1(sp_, wa, B) * len(B) + f1(sp_, wb, A) * len(A)) / n          # weights chosen on one half, scored on the other
    return w, best, singles, cross


def write_pred(path, qids, cids, S):
    pred = dict(zip(qids, topk_ids(S, cids)))
    for q, v in pred.items():
        assert len(v) == K and len(set(v)) == K and q not in v
    os.makedirs(os.path.dirname(path), exist_ok=True)
    json.dump(pred, open(path, "w"))
    return pred


def zip_pred(pred_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")


def main(argv=None):
    """ENSEMBLE: lexical signals + the dense scores of several finished runs, weights tuned on dev."""
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--runs", required=True, help="comma-separated run names (their dense_scores.npz)")
    ap.add_argument("--name", default="ens_all")
    ap.add_argument("--hf_repo", default="")
    a = ap.parse_args(argv)
    runs = [r for r in a.runs.split(",") if r]
    dv, te = Split(a.data_dir, "dev"), Split(a.data_dir, "test")
    names = ["bm25u", "bm25b"]
    for r in runs:
        z = np.load(os.path.join(a.out_dir, r, "dense_scores.npz"))
        for kind in ["mean", "max"]:
            dv.add(f"{r}:{kind}", z[f"dev_{kind}"]); te.add(f"{r}:{kind}", z[f"test_{kind}"]); names.append(f"{r}:{kind}")
    w, best, singles, cross = tune_report(dv, names, require=[n for n in names if ":" in n])
    rd = os.path.join(a.out_dir, a.name); os.makedirs(rd, exist_ok=True)
    write_pred(os.path.join(rd, "dev_prediction.json"), dv.qids, dv.cids, score(dv, w))
    write_pred(os.path.join(rd, "prediction.json"), te.qids, te.cids, score(te, w))
    zip_pred(os.path.join(rd, "prediction.json"), os.path.join(rd, "submission.zip"))
    info = {"run": a.name, "members": runs, "weights": w, "dev_f1": best, "dev_f1_2fold": cross, "single_signal_dev_f1": singles}
    json.dump(info, open(os.path.join(rd, "summary.json"), "w"), indent=1)
    print(f"[ensemble] {a.name}: DEV Micro-F1@7 {best:.4f} (2-fold check {cross:.4f}) | weights "
          + ", ".join(f"{k}={v:g}" for k, v in w.items() if v), flush=True)
    if a.hf_repo:
        from p7ret import hf_up
        for fn in ["prediction.json", "submission.zip", "dev_prediction.json", "summary.json"]:
            hf_up(a.hf_repo, os.path.join(rd, fn), f"phase7/runs/{a.name}/{fn}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p7fuse.py


In [5]:
%%writefile /kaggle/working/p7ret.py
"""CS-781 Phase 7: Prior Case Retrieval with our Phase-1 legal LM as a dense retriever (bi-encoder).

case -> windows of `chunk_len` tokens (our tokenizer) -> Phase-1 model -> final norm -> mean over tokens -> unit vector.
Long cases: up to `max_chunks` windows spread evenly over the whole case; case vector = mean of its window vectors
("mean"); also MaxSim ("max": every query window takes its best-matching candidate window, averaged).
Training (train split only): contrastive InfoNCE - query window vs a relevant candidate window, against in-batch candidates
and BM25 hard negatives (other relevant candidates of the same query are masked, never used as negatives).
Methods: lorma (compulsory) | full | partial (top layers only) | ia3 | none (Phase-1 as it is) ; --kd_teacher = knowledge
distillation (student copies the teacher's similarity distribution over the same batch).
Selection on DEV with the official metric (Micro-F1@7); final ranking = these dense scores fused with BM25 (p7fuse).
"""
import argparse, json, math, os, random, subprocess, sys, time, zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig
from p7data import topk_ids, micro_f1, K
import p7fuse as FU


# ================================================================================================================ adapters
class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi 2025), additive rank inflation: W' = (I + (alpha/r) B A) W0 ; B = 0 at start."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        d = base.out_features
        self.A = nn.Parameter(torch.randn(r, d) / math.sqrt(d)); self.B = nn.Parameter(torch.zeros(d, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class IA3Out(nn.Module):
    """IA3 (Liu et al. 2022): learned vector that rescales the output of k / v (attention)."""

    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """IA3 on the feed-forward: rescale the inner activations before the down projection."""

    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def setup_method(lm, a):
    for p in lm.parameters():
        p.requires_grad = a["method"] == "full"
    m = a["method"]
    if m in ("full", "none"):
        return
    if m == "partial":
        blocks = list(lm.blocks)
        for blk in blocks[-a["partial_layers"]:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
        return
    n = 0
    for mod in list(lm.blocks.modules()):
        for name, ch in list(mod.named_children()):
            if not isinstance(ch, nn.Linear):
                continue
            if m == "lorma" and name in a["targets"].split(","):
                setattr(mod, name, LoRMALinear(ch, a["rank"], a["alpha"])); n += 1
            elif m == "ia3" and name in ("k", "v"):
                setattr(mod, name, IA3Out(ch)); n += 1
            elif m == "ia3" and name == "down":
                setattr(mod, name, IA3In(ch)); n += 1
    print(f"{m}: wrapped {n} layers", flush=True)


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ================================================================================================================ encoding
def embed(lm, x, lens):
    """x (B,T) right-padded token ids, lens (B,) -> (B,d) unit vectors (final norm + mean over the real tokens)."""
    h = lm._embed(x)
    for blk in lm._decoder_layers():
        h = blk(h, rope=lm._rope(), pos=0, causal=True)
    h = lm.norm(h).float()
    m = (torch.arange(x.shape[1], device=x.device)[None, :] < lens[:, None]).float()
    e = (h * m[..., None]).sum(1) / lens[:, None].float()
    return F.normalize(e, dim=-1)


class Docs:
    def __init__(self, tok, off, ids):
        self.tok, self.off = tok, off
        self.ids = [str(x) for x in ids]; self.index = {c: i for i, c in enumerate(self.ids)}

    def doc(self, i):
        return self.tok[self.off[i]:self.off[i + 1]]

    def windows(self, i, L, M):
        """up to M full-length windows spread evenly over the case (the last window ends at the end)."""
        d = self.doc(i); n = len(d)
        if n <= L:
            return [d if n else np.zeros(1, np.uint16)]
        nw = math.ceil(n / L)
        starts = [min(k * L, n - L) for k in range(nw)] if nw <= M else [int(round(s)) for s in np.linspace(0, n - L, M)]
        return [d[s:s + L] for s in starts]

    def random_window(self, i, L, rng):
        d = self.doc(i); n = len(d)
        if n <= L:
            return d if n else np.zeros(1, np.uint16)
        s = rng.randint(0, n - L)
        return d[s:s + L]


def batch_tensor(wins, dev):
    T = max(len(w) for w in wins)
    x = np.zeros((len(wins), T), np.int64); lens = np.zeros(len(wins), np.int64)
    for i, w in enumerate(wins):
        x[i, :len(w)] = w; lens[i] = len(w)
    return torch.as_tensor(x, device=dev), torch.as_tensor(lens, device=dev)


@torch.no_grad()
def encode_all(lm, docs, L, M, dev, amp, bs=32):
    """-> (n_docs, M_used, d) float16 window vectors + (n_docs,) number of windows."""
    lm.eval()
    wins, owner = [], []
    for i in range(len(docs.ids)):
        for w in docs.windows(i, L, M):
            wins.append(w); owner.append(i)
    order = np.argsort([-len(w) for w in wins], kind="stable")
    E = np.zeros((len(wins), lm.cfg.d_model), np.float16)
    for k in range(0, len(order), bs):
        sel = order[k:k + bs]
        x, lens = batch_tensor([wins[j] for j in sel], dev)
        with amp():
            E[sel] = embed(lm, x, lens).cpu().numpy().astype(np.float16)
    owner = np.array(owner)
    nw = np.bincount(owner, minlength=len(docs.ids))
    Mu = int(nw.max())
    W = np.zeros((len(docs.ids), Mu, E.shape[1]), np.float16)
    pos = np.zeros(len(docs.ids), np.int64)
    for j, o in enumerate(owner):
        W[o, pos[o]] = E[j]; pos[o] += 1
    return W, nw


@torch.no_grad()
def similarities(Wq, nq, Wc, nc, dev):
    """-> mean-pooled cosine (Q,C) and MaxSim (Q,C)."""
    q = torch.as_tensor(Wq, device=dev).float(); c = torch.as_tensor(Wc, device=dev).float()
    mq = (torch.arange(q.shape[1], device=dev)[None] < torch.as_tensor(nq, device=dev)[:, None]).float()
    mc = (torch.arange(c.shape[1], device=dev)[None] < torch.as_tensor(nc, device=dev)[:, None]).float()
    qm = F.normalize((q * mq[..., None]).sum(1), dim=-1); cm = F.normalize((c * mc[..., None]).sum(1), dim=-1)
    S_mean = (qm @ cm.T).cpu().numpy()
    cf = c.reshape(-1, c.shape[-1]); S_max = np.zeros_like(S_mean)
    for i in range(q.shape[0]):
        s = (q[i, :int(nq[i])] @ cf.T).view(int(nq[i]), c.shape[0], c.shape[1])
        s = s.masked_fill(mc[None] == 0, -2.0).max(-1).values.mean(0)
        S_max[i] = s.cpu().numpy()
    return S_mean, S_max


# ================================================================================================================ main
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", default="dapt_llama")
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="lorma", choices=["lorma", "full", "partial", "ia3", "none"])
    p.add_argument("--kd_teacher", default="")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the label loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=1.0, help="extra temperature for the distillation softmax")
    p.add_argument("--chunk_len", type=int, default=512)
    p.add_argument("--max_chunks", type=int, default=8, help="windows per case at inference")
    p.add_argument("--val_chunks", type=int, default=2, help="windows per case for the quick dev check after each epoch")
    p.add_argument("--bs", type=int, default=8, help="queries per step (each + 1 relevant + 1 hard negative)")
    p.add_argument("--tau", type=float, default=0.05)
    p.add_argument("--epochs", type=float, default=20)
    p.add_argument("--max_train_min", type=float, default=35)
    p.add_argument("--lr", type=float, default=2e-5, help="full / partial fine-tuning")
    p.add_argument("--peft_lr", type=float, default=3e-4, help="LoRMA")
    p.add_argument("--ia3_lr", type=float, default=3e-3)
    p.add_argument("--partial_layers", type=int, default=4)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o,gate,up,down")
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--seed", type=int, default=7)
    p.add_argument("--data_dir", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1)
    p.add_argument("--limit", type=int, default=0, help="debug: first N queries / candidates of every split")
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for k in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo, repo_type="model"); return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (k + 1))


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def load_lm(args, base_run, base_file, margs):
    p = os.path.join(args.ckpt_dir, "runs", base_run, base_file)
    if not os.path.exists(p):
        hf_get(args.hf_repo, f"runs/{base_run}/{base_file}", args.ckpt_dir)
    ck = torch.load(p, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    assert cfg["arch"] != "encdec", "retrieval here uses the decoder-only Phase-1 models"
    lm = LegalLM(ModelConfig(**cfg))
    missing, _ = lm.load_state_dict(ck["model"], strict=False)
    assert not missing, missing[:5]
    setup_method(lm, margs)
    return lm, cfg


def load_run(args, run, lm):
    p = os.path.join(args.out_dir, run, "best.pt")
    if not os.path.exists(p):
        p = hf_get(args.hf_repo, f"phase7/runs/{run}/best.pt", os.path.join(args.out_dir, "_hub", run))
    ck = torch.load(p, map_location="cpu", weights_only=False)
    cur = lm.state_dict()
    assert set(ck["state_dict"]) <= set(cur), f"{run}: weights do not match"
    cur.update({k: v.to(cur[k].dtype) for k, v in ck["state_dict"].items()})
    lm.load_state_dict(cur)
    return ck["args"]


def main(argv=None):
    args = parse_args(argv)
    t_start = time.time()
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu"); cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    rd = os.path.join(args.out_dir, args.run_name); os.makedirs(rd, exist_ok=True)
    logf = open(os.path.join(rd, "log.txt"), "a")
    work = os.path.dirname(os.path.abspath(__file__))

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    def load_split(s):
        z = np.load(os.path.join(args.data_dir, f"{s}_tok.npz"))
        q = Docs(z["q_tok"], z["q_off"], z["q_ids"]); c = Docs(z["c_tok"], z["c_off"], z["c_ids"])
        if args.limit:                                       # debug / safety test: a few queries + THEIR relevant cases
            q.ids = q.ids[:args.limit]
            lab = os.path.join(args.data_dir, f"labels_{s}.json")
            keep = set()
            if os.path.exists(lab):
                g = json.load(open(lab))
                for x in q.ids:
                    keep.update(g.get(x, []))
            keep = [x for x in c.ids if x in keep] + c.ids[:max(args.limit * 4, 50)]
            c.ids = list(dict.fromkeys(keep))
            q.index = {x: i for i, x in enumerate(q.ids)}; c.index = {x: i for i, x in enumerate(c.ids)}
        return q, c

    lm, cfg = load_lm(args, args.base_run, args.base_file, vars(args))
    lm.to(dev)
    total, trainable = count_params(lm)
    log(f"backbone {args.base_run} ({cfg['arch']}) | method {args.method}{' + KD from ' + args.kd_teacher if args.kd_teacher else ''}"
        f" | windows {args.chunk_len} tokens, {args.max_chunks} per case | params {total / 1e6:.1f}M trainable {trainable / 1e6:.3f}M")
    L = min(args.chunk_len, cfg["max_seq_len"])
    trq, trc = load_split("train"); dvq, dvc = load_split("dev"); teq, tec = load_split("test")
    gold_tr = json.load(open(os.path.join(args.data_dir, "labels_train.json")))
    gold_dv = json.load(open(os.path.join(args.data_dir, "labels_dev.json")))
    hard = json.load(open(os.path.join(args.data_dir, "hard_train.json")))

    def dev_f1(M):
        Wq, nq = encode_all(lm, dvq, L, M, dev, amp); Wc, nc = encode_all(lm, dvc, L, M, dev, amp)
        Sm, Sx = similarities(Wq, nq, Wc, nc, dev)
        for i, q in enumerate(dvq.ids):
            if q in dvc.index:
                Sm[i, dvc.index[q]] = -9; Sx[i, dvc.index[q]] = -9
        g = {q: gold_dv[q] for q in dvq.ids}
        return micro_f1(dict(zip(dvq.ids, topk_ids(Sm, dvc.ids))), g)[0], micro_f1(dict(zip(dvq.ids, topk_ids(Sx, dvc.ids))), g)[0]

    # ---------------- training ----------------
    best = {"dev": -1.0, "epoch": 0}
    steps, ep, train_min = 0, 0, 0.0
    names = [n for n, p in lm.named_parameters() if p.requires_grad]
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    if args.method != "none":
        teacher = None
        if args.kd_teacher:
            tp = os.path.join(args.out_dir, args.kd_teacher, "best.pt")
            if not os.path.exists(tp):
                tp = hf_get(args.hf_repo, f"phase7/runs/{args.kd_teacher}/best.pt", os.path.join(args.out_dir, "_hub", args.kd_teacher))
            ta = torch.load(tp, map_location="cpu", weights_only=False)["args"]
            tns = argparse.Namespace(**{**vars(args), **{k: ta[k] for k in ["method", "rank", "alpha", "targets", "partial_layers"]}})
            teacher, tcfg = load_lm(tns, ta["base_run"], ta["base_file"], vars(tns))
            load_run(args, args.kd_teacher, teacher)
            teacher.to(dev).eval()
            for p_ in teacher.parameters():
                p_.requires_grad = False
            log(f"KD teacher {args.kd_teacher} ({tcfg['arch']}, {count_params(teacher)[0] / 1e6:.1f}M) -> student {cfg['arch']} {total / 1e6:.1f}M")
        lr = {"full": args.lr, "partial": args.lr, "lorma": args.peft_lr, "ia3": args.ia3_lr}[args.method]
        params = [p_ for p_ in lm.parameters() if p_.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=args.wd)
        scaler = torch.amp.GradScaler("cuda", enabled=cuda)
        train_q = [q for q in trq.ids if any(c in trc.index for c in gold_tr.get(q, []))]
        assert len(train_q) >= 2, "fewer than 2 training queries have a relevant case in the pool"
        args.bs = min(args.bs, len(train_q))
        rng = random.Random(args.seed)
        spe = max(1, len(train_q) // args.bs)
        total_steps = int(spe * args.epochs) if args.max_steps <= 0 else args.max_steps
        warm = max(1, int(0.05 * total_steps)); budget = args.max_train_min * 60
        log(f"train {len(train_q)} queries with relevant cases in the pool | {spe} steps/epoch | lr {lr} | budget {args.max_train_min:.0f} min")
        f_mean, f_max = dev_f1(args.val_chunks)
        log(f"before training: DEV Micro-F1@7 (dense only, {args.val_chunks} windows) mean {f_mean:.4f} | max {f_max:.4f}")
        best.update(dev=max(f_mean, f_max), epoch=0, zero_shot=max(f_mean, f_max))
        sd = lm.state_dict(); best_state = {k: sd[k].detach().cpu().clone() for k in names}
        t0, stop = time.time(), False
        while not stop:
            ep += 1; lm.train(); rng.shuffle(train_q); oom = False
            for k in range(0, len(train_q) - args.bs + 1, args.bs):
                qs = train_q[k:k + args.bs]
                pos = [rng.choice([c for c in gold_tr[q] if c in trc.index]) for q in qs]
                neg = [rng.choice([c for c in hard.get(q, []) if c in trc.index] or trc.ids) for q in qs]
                cand = pos + neg
                wins = [trq.random_window(trq.index[q], L, rng) for q in qs] + [trc.random_window(trc.index[c], L, rng) for c in cand]
                x, lens = batch_tensor(wins, dev)
                mask = torch.zeros(len(qs), len(cand), dtype=torch.bool, device=dev)       # True = do not use as a negative
                for i, q in enumerate(qs):
                    rel = set(gold_tr[q])
                    for j, c in enumerate(cand):
                        if j != i and (c in rel or c == q or c == pos[i]):
                            mask[i, j] = True
                lr_mult = min(1.0, (steps + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, steps / total_steps)))
                for g in opt.param_groups:
                    g["lr"] = lr * max(lr_mult, 0.02)
                opt.zero_grad(set_to_none=True)
                try:
                    with amp():
                        e = embed(lm, x, lens)
                    s = (e[:len(qs)] @ e[len(qs):].T) / args.tau
                    s = s.masked_fill(mask, -1e4)
                    lab = torch.arange(len(qs), device=dev)
                    loss = F.cross_entropy(s, lab)
                    if teacher is not None:
                        with torch.no_grad(), amp():
                            te = embed(teacher, x, lens)
                        ts = ((te[:len(qs)] @ te[len(qs):].T) / args.tau).masked_fill(mask, -1e4)
                        kd = F.kl_div(F.log_softmax(s / args.kd_T, -1), F.softmax(ts / args.kd_T, -1), reduction="batchmean") * args.kd_T ** 2
                        loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
                    scaler.scale(loss).backward()
                except torch.cuda.OutOfMemoryError:
                    opt.zero_grad(set_to_none=True); e = s = loss = x = None
                    torch.cuda.empty_cache()
                    if args.bs <= 2:
                        raise
                    args.bs //= 2; oom = True
                    log(f"out of GPU memory -> {args.bs} queries per step"); break
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
                steps += 1
                if steps == 20 and args.max_steps <= 0:
                    sec = (time.time() - t0) / steps; fit = int(0.85 * budget / sec)
                    log(f"{sec:.2f} s/step -> {spe * sec / 60:.1f} min/epoch")
                    if fit < total_steps:
                        total_steps = max(fit, 50); log(f"time budget allows {total_steps} steps ({total_steps / spe:.1f} epochs)")
                if steps % 50 == 0:
                    log(f"epoch {ep} step {steps}/{total_steps} loss {loss.item():.4f} | {(time.time() - t0) / 60:.1f} min")
                if steps >= total_steps or time.time() - t0 > budget:
                    stop = True; break
            if oom:
                ep -= 1; continue
            if ep > 1000:                                    # can never loop for ever
                stop = True
            f_mean, f_max = dev_f1(args.val_chunks)
            log(f"epoch {ep} (step {steps}): DEV Micro-F1@7 dense only - mean {f_mean:.4f} | max {f_max:.4f} | {(time.time() - t0) / 60:.1f} min")
            if max(f_mean, f_max) > best["dev"]:
                best.update(dev=max(f_mean, f_max), epoch=ep)
                sd = lm.state_dict(); best_state = {k: sd[k].detach().cpu().clone() for k in names}
        train_min = (time.time() - t0) / 60
        sd = lm.state_dict(); sd.update(best_state); lm.load_state_dict(sd)
        teacher = opt = None
        if cuda:
            torch.cuda.empty_cache()
        log(f"training done: {steps} steps, {ep} epochs, {train_min:.1f} min | best quick DEV F1 {best['dev']:.4f} (epoch {best['epoch']})")
        sd = lm.state_dict()
        torch.save({"state_dict": {k: sd[k].half() if sd[k].is_floating_point() else sd[k] for k in names}, "args": vars(args),
                    "lm_config": cfg}, os.path.join(rd, "best.pt"))
        hf_up(args.hf_repo, os.path.join(rd, "best.pt"), f"phase7/runs/{args.run_name}/best.pt")

    # ---------------- final encoding (all windows) + scores ----------------
    t1 = time.time()
    out = {}
    for s, (q, c) in {"dev": (dvq, dvc), "test": (teq, tec)}.items():
        Wq, nq = encode_all(lm, q, L, args.max_chunks, dev, amp); Wc, nc = encode_all(lm, c, L, args.max_chunks, dev, amp)
        out[f"{s}_mean"], out[f"{s}_max"] = similarities(Wq, nq, Wc, nc, dev)
    enc_s = time.time() - t1
    n_docs = len(dvq.ids) + len(dvc.ids) + len(teq.ids) + len(tec.ids)
    np.savez_compressed(os.path.join(rd, "dense_scores.npz"), **{k: v.astype(np.float16) for k, v in out.items()})

    # dense only + fused with the lexical signals (weights chosen on dev)
    if args.limit:
        log("debug run (--limit): skipping fusion"); return
    dv, te = FU.Split(args.data_dir, "dev"), FU.Split(args.data_dir, "test")
    for kind in ["mean", "max"]:
        dv.add(f"dense_{kind}", out[f"dev_{kind}"]); te.add(f"dense_{kind}", out[f"test_{kind}"])
    res = {}
    for kind in ["mean", "max"]:
        res[f"dense_{kind}"] = FU.f1(dv, {f"dense_{kind}": 1.0})
    dk = max(["mean", "max"], key=lambda k: res[f"dense_{k}"])
    FU.write_pred(os.path.join(rd, "dev_prediction_dense.json"), dv.qids, dv.cids, FU.score(dv, {f"dense_{dk}": 1.0}))
    FU.write_pred(os.path.join(rd, "prediction_dense.json"), te.qids, te.cids, FU.score(te, {f"dense_{dk}": 1.0}))
    FU.zip_pred(os.path.join(rd, "prediction_dense.json"), os.path.join(rd, "submission_dense.zip"))
    w, fbest, singles, cross = FU.tune_report(dv, ["bm25u", "bm25b", "dense_mean", "dense_max"], require=["dense_mean", "dense_max"])
    FU.write_pred(os.path.join(rd, "dev_prediction.json"), dv.qids, dv.cids, FU.score(dv, w))
    FU.write_pred(os.path.join(rd, "prediction.json"), te.qids, te.cids, FU.score(te, w))
    FU.zip_pred(os.path.join(rd, "prediction.json"), os.path.join(rd, "submission.zip"))

    def official(pred_file):
        r = subprocess.run([sys.executable, os.path.join(work, "evaluate_phase7.py"), "--truth", os.path.join(args.data_dir, "labels_dev.json"),
                            "--prediction", pred_file, "--candidate-ids", os.path.join(args.data_dir, "dev_candidate_ids.json")],
                           capture_output=True, text=True)
        txt = r.stdout + r.stderr
        for line in txt.splitlines():
            if line.startswith("Micro-F1@7"):
                return float(line.split(":")[1]), " | ".join(l for l in txt.splitlines() if l.startswith(("Micro", "TP", "LOCAL")))
        return None, txt[-500:]
    of_dense, _ = official(os.path.join(rd, "dev_prediction_dense.json"))
    of_fused, of_txt = official(os.path.join(rd, "dev_prediction.json"))
    log(f"DEV (official evaluator): dense only ({dk}) Micro-F1@7 {of_dense} | fused with BM25 {of_fused} (2-fold check {cross:.4f}) | "
        f"weights " + ", ".join(f"{k}={v:g}" for k, v in w.items() if v))
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": cfg["arch"], "method": args.method, "kd_teacher": args.kd_teacher,
               "params_total": total, "params_trainable": trainable, "dev_f1_dense_mean": res["dense_mean"], "dev_f1_dense_max": res["dense_max"],
               "dev_f1_dense": of_dense, "dev_f1": of_fused, "dev_f1_2fold": cross, "evaluator": of_txt, "fusion_weights": w,
               "single_signal_dev_f1": singles, "zero_shot_quick_dev": best.get("zero_shot"), "best_epoch": best["epoch"],
               "epochs_run": ep, "steps": steps, "train_minutes": round(train_min, 1),
               "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else 0.0,
               "encode_seconds_per_case": round(enc_s / max(1, n_docs), 4), "args": vars(args)}
    json.dump(summary, open(os.path.join(rd, "summary.json"), "w"), indent=1)
    for fn in ["prediction.json", "submission.zip", "dev_prediction.json", "prediction_dense.json", "submission_dense.zip",
               "dev_prediction_dense.json", "dense_scores.npz", "log.txt", "summary.json"]:
        hf_up(args.hf_repo, os.path.join(rd, fn), f"phase7/runs/{args.run_name}/{fn}")
    log(f"DONE in {(time.time() - t_start) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p7ret.py


In [6]:
%%writefile /kaggle/working/evaluate_phase7.py
#!/usr/bin/env python3
"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).

It uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /
Micro-Recall@7 definitions as the Codabench scorer.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

K = 7


def reject_constant(value: str) -> None:
    raise ValueError(f"Non-standard JSON numeric value '{value}' is not allowed.")


def pairs_no_duplicates(pairs):
    result = {}
    duplicates = []
    for key, value in pairs:
        if key in result:
            duplicates.append(key)
        result[key] = value
    if duplicates:
        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")
    return result


def load_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"
        ) from exc


def load_truth(path: Path) -> dict[str, list[str]]:
    data = load_json(path)
    # Accept either the simple gold.json mapping or a Phase-7 dev query file:
    # {query_id: {text: [...], relevant_candidates: [...]}}
    if not isinstance(data, dict) or not data:
        raise ValueError("Truth must be a non-empty JSON object.")
    out: dict[str, list[str]] = {}
    for query_id, value in data.items():
        if isinstance(value, dict) and "relevant_candidates" in value:
            labels = value["relevant_candidates"]
        else:
            labels = value
        if not isinstance(query_id, str) or not query_id.strip():
            raise ValueError("Truth contains an empty/invalid query ID.")
        if not isinstance(labels, list):
            raise ValueError(f"Truth for query '{query_id}' must contain a list of relevant candidate IDs.")
        cleaned=[]; seen=set()
        for idx,candidate_id in enumerate(labels):
            if not isinstance(candidate_id,str) or not candidate_id.strip():
                # Treat blank source labels as absent labels, matching the builder.
                continue
            if candidate_id in seen:
                raise ValueError(f"Truth for query '{query_id}' contains duplicate candidate ID '{candidate_id}'.")
            seen.add(candidate_id); cleaned.append(candidate_id)
        out[query_id]=cleaned
    return out


def load_candidate_ids(path: Path) -> set[str]:
    data = load_json(path)
    if not isinstance(data, list) or not data:
        raise ValueError("candidate_ids must be a non-empty JSON list.")
    ids=[]; seen=set()
    for idx,v in enumerate(data):
        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")
        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID '{v}'.")
        seen.add(v); ids.append(v)
    return set(ids)


def load_prediction(path: Path) -> Any:
    return load_json(path)


def validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:
    errors=[]
    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]
    gold_ids=set(gold); pred_ids=set(pred)
    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID '{q}'.")
    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID '{q}'.")
    for q in sorted(gold_ids & pred_ids):
        vals=pred[q]
        if not isinstance(vals,list):
            errors.append(f"Query '{q}': prediction must be a list."); continue
        if len(vals)!=K: errors.append(f"Query '{q}': expected exactly {K} candidates, got {len(vals)}.")
        seen=set()
        for i,c in enumerate(vals):
            if not isinstance(c,str) or not c.strip(): errors.append(f"Query '{q}': index {i} is not a non-empty string."); continue
            if c in seen: errors.append(f"Query '{q}': duplicate candidate ID '{c}'.")
            seen.add(c)
            if c not in candidates: errors.append(f"Query '{q}': candidate ID '{c}' is outside the candidate pool.")
    return errors


def score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:
    tp=fp=fn=0
    for q,labels in gold.items():
        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")
    p.add_argument("--prediction",type=Path,required=True)
    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")
    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")
    return p.parse_args()


def main():
    a=parse_args()
    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)
    errs=validate(pred,gold,candidates)
    if errs:
        print("LOCAL VALIDATION FAILED")
        for i,e in enumerate(errs,1): print(f"  {i}. {e}")
        print(f"Total validation errors: {len(errs)}")
        raise SystemExit(1)
    result=score(gold,pred)
    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}
    print("Validation passed.")
    print(f"Micro-F1@7: {scores['primary_metric_1']:.10f}")
    print(f"Micro-Precision@7: {scores['secondary_metric_1']:.10f}")
    print(f"Micro-Recall@7: {scores['secondary_metric_2']:.10f}")
    print(f"TP={result['tp']} FP={result['fp']} FN={result['fn']}")
    if a.output:
        a.output.parent.mkdir(parents=True,exist_ok=True)
        with a.output.open('w',encoding='utf-8') as f: json.dump(scores,f,indent=2); f.write('\n')
        print(f"Wrote {a.output}")

if __name__=='__main__': main()


Writing /kaggle/working/evaluate_phase7.py


In [7]:
# ===================== P7 CELL 3 - Data: clean + tokenise once with our tokenizer, BM25 / TF-IDF / years (cached on HF) =====================
sys.path.insert(0, WORK)
hub_files = set(api.list_repo_files(CKPT_REPO))
cache = [f"{s}_tok.npz" for s in ["train", "dev", "test"]] + [f"lex_{s}.npz" for s in ["train", "dev", "test"]] + \
        [f"tfidf_{s}.npz" for s in ["train", "dev", "test"]] + [f"{s}_candidate_ids.json" for s in ["train", "dev", "test"]] + \
        ["labels_train.json", "labels_dev.json", "hard_train.json", "stats.json"]
if all(f"phase7/data/{f}" in hub_files for f in cache):
    for f in cache:
        shutil.copy(hf_hub_download(CKPT_REPO, f"phase7/data/{f}", repo_type="model", token=HF_TOKEN, local_dir=f"{P7}/hub"), f"{DATA}/{f}")
    print("data cache downloaded from Hugging Face")
else:
    t0 = time.time()
    for p in [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo="Phase_7_PCR", recursive=True)
              if getattr(x, "size", None) is not None]:
        lp = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P7}/raw")
        shutil.copy(lp, f"{RAW}/{p.split('/')[-1]}")
    need = ["train.json", "train_candidates.json", "dev.json", "dev_candidates.json", "test.json", "test_candidates.json"]
    missing = [n for n in need if not os.path.exists(f"{RAW}/{n}")]
    assert not missing, f"missing files {missing} (found {sorted(os.listdir(RAW))}) - paste this output to Claude"
    print("Phase-7 files:", {f: f"{os.path.getsize(f'{RAW}/{f}') / 1e6:.1f} MB" for f in sorted(os.listdir(RAW))}, f"| {time.time() - t0:.0f}s")
    r = subprocess.run([sys.executable, f"{WORK}/p7data.py", "--raw_dir", RAW, "--tok", TOK_PATH, "--out_dir", DATA], capture_output=True, text=True)
    print(r.stdout[-4000:], r.stderr[-3000:])
    assert r.returncode == 0, "data preparation failed - paste this output to Claude"
    for f in cache:
        api.upload_file(path_or_fileobj=f"{DATA}/{f}", path_in_repo=f"phase7/data/{f}", repo_id=CKPT_REPO, repo_type="model")
    print(f"prepared + uploaded in {(time.time() - t0) / 60:.1f} min")
stats = json.load(open(f"{DATA}/stats.json"))
for s in ["train", "dev", "test"]:
    print(s, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in stats[s].items()})
print("BM25 settings (chosen on train):", stats["bm25"])


phase7/data/train_tok.npz: reconstructing file:   0%|          |  0.00B / 82.4MB            

phase7/data/train_tok.npz: downloading bytes:           |  0.00B            

phase7/data/dev_tok.npz: reconstructing file:   0%|          |  0.00B / 24.5MB            

phase7/data/dev_tok.npz: downloading bytes:           |  0.00B            

phase7/data/test_tok.npz: reconstructing file:   0%|          |  0.00B / 34.3MB            

phase7/data/test_tok.npz: downloading bytes:           |  0.00B            

... (16 outputs left out)

labels_dev.json:   0%|          | 0.00/15.5k [00:00<?, ?B/s]

hard_train.json:   0%|          | 0.00/592k [00:00<?, ?B/s]

stats.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

data cache downloaded from Hugging Face
train {'queries': 827, 'candidates': 4320, 'query_tokens_mean': 8525.1294, 'query_tokens_median': 5655.0, 'cand_tokens_mean': 7879.5979, 'cand_tokens_median': 4549.5, 'queries_inside_pool': 827, 'year_found_queries': 0.9976, 'year_found_cands': 0.9806, 'relevant_mean': 6.4184, 'relevant_median': 4.0, 'relevant_max': 52, 'relevant_in_pool_share': 1.0, 'queries_without_relevant': 10, 'F1@7_bm25u': 0.2074, 'F1@7_bm25b': 0.3498, 'F1@7_bm25u+b': 0.3325}
dev {'queries': 118, 'candidates': 1023, 'query_tokens_mean': 10489.2458, 'query_tokens_median': 5614.5, 'cand_tokens_mean': 10743.4086, 'cand_tokens_median': 5046.0, 'queries_inside_pool': 118, 'year_found_queries': 1.0, 'year_found_cands': 0.9922, 'relevant_mean': 8.2458, 'relevant_median': 5.0, 'relevant_max': 92, 'relevant_in_pool_share': 1.0, 'queries_without_relevant': 0, 'F1@7_bm25u': 0.2357, 'F1@7_bm25b': 0.4369, 'F1@7_bm25u+b': 0.4169}
test {'queries': 237, 'candidates': 1727, 'query_tokens_me

In [8]:
# ===================== P7 CELL 7 - Experiments on both GPUs (queue with KD dependency, resumable) =====================
# Each run: fine-tune a Phase-1 model as a retriever on TRAIN (time budget, best epoch kept by a quick DEV check), encode the
# dev/test cases, score them with the OFFICIAL evaluate_phase7.py (dense only, and fused with BM25), write submission.zip.
# Finished runs (phase7/runs/<run>/summary.json on HF) are skipped and downloaded.
CK = "/tmp/cs781/ckpt"
OUT = f"{WORK}/p7runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(chunk_len=512, max_chunks=8, val_chunks=2, bs=8, tau=0.05, epochs=20, max_train_min=35)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
TEACHER = "p7_llama_lorma"
RUNS = [   # priority order (2 GPUs take jobs from the top)
    (TEACHER,              {**LLAMA,  "method": "lorma", "peft_lr": 3e-4}),           # LoRMA (compulsory) + KD teacher
    ("p7_llama_full",      {**LLAMA,  "method": "full", "lr": 2e-5}),                 # full fine-tuning
    ("p7_llama_ia3",       {**LLAMA,  "method": "ia3", "ia3_lr": 3e-3}),              # IA3
    ("p7_llama_partial",   {**LLAMA,  "method": "partial", "lr": 5e-5, "partial_layers": 4}),   # partial / layer-wise (top 4 layers)
    ("p7_shared_kd",       {**SHARED, "method": "full", "lr": 5e-5, "kd_teacher": TEACHER}),    # KD: llama 108M -> shared 59M
    ("p7_llama_zeroshot",  {**LLAMA,  "method": "none"}),                             # Phase-1 model with NO fine-tuning (transfer)
    ("p7_shared_full",     {**SHARED, "method": "full", "lr": 5e-5}),                 # the same student WITHOUT KD
]
ONLY = None

def p7_cmd(name, cfg, out_dir, hf_repo, extra=()):
    cmd = [sys.executable, f"{WORK}/p7ret.py", "--run_name", name, "--data_dir", DATA, "--ckpt_dir", CK, "--out_dir", out_dir, "--hf_repo", hf_repo]
    for k, v in {**COMMON, **cfg}.items():
        cmd += [f"--{k}", str(v)]
    return cmd + list(extra)

# ---- safety test on the REAL data: 3 steps, small subset, nothing uploaded ----
t_s = time.time()
try:
    r = subprocess.run(p7_cmd("_smoke", {**SHARED, "method": "lorma", "max_steps": 3, "bs": 2, "max_chunks": 2}, f"{P7}/smoke", "", ["--limit", "6"]),
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True, timeout=900)
except subprocess.TimeoutExpired as e:
    print(((e.stdout or b"").decode() if isinstance(e.stdout, bytes) else (e.stdout or ""))[-3000:])
    raise RuntimeError("SAFETY TEST TOOK > 15 min - nothing was trained. Send this output to Claude.")
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-6:]))
if r.returncode != 0 or not os.path.exists(f"{P7}/smoke/_smoke/dense_scores.npz"):
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
# the fusion + official evaluator path on the lexical signals alone (no model): also the BM25 reference score
import p7fuse as FU
_dv = FU.Split(DATA, "dev"); _w, _f, _sg, _cx = FU.tune_report(_dv, ["bm25u", "bm25b"])
FU.write_pred(f"{P7}/smoke/bm25_dev.json", _dv.qids, _dv.cids, FU.score(_dv, _w))
r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase7.py", "--truth", f"{DATA}/labels_dev.json", "--prediction", f"{P7}/smoke/bm25_dev.json",
                    "--candidate-ids", f"{DATA}/dev_candidate_ids.json"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout + r.stderr
print(f"safety test passed in {time.time() - t_s:.0f}s | lexical only (BM25 words+pairs, year, hub, feedback; no model): DEV "
      + " | ".join(l for l in r.stdout.splitlines() if l.startswith("Micro")) + f" | weights { {k: v for k, v in _w.items() if v} }\n")

hub_files = set(api.list_repo_files(CKPT_REPO))
done, failed, running, lock = set(), set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
for n, c in list(pending):
    if f"phase7/runs/{n}/summary.json" in hub_files:
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json", "dev_prediction.json", "dense_scores.npz", "log.txt",
                   "submission_dense.zip"]:
            if f"phase7/runs/{n}/{fn}" in hub_files:
                shutil.copy(hf_hub_download(CKPT_REPO, f"phase7/runs/{n}/{fn}", repo_type="model", local_dir=f"{P7}/hub", token=HF_TOKEN), f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = c.get("kd_teacher")
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                if f"phase7/runs/{t}/best.pt" in hub_files: done.add(t)
                else: failed.add(t)
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: teacher {t} failed", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "step" in l or "DEV" in l or "Error" in l or "DONE" in l]
    return keep[-1][:200] if keep else "(starting)"

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(p7_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu} | {status_line(name)}", flush=True)
            if not ok_:
                print("\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

if torch.cuda.device_count() == 0:
    print("CPU session: switch Accelerator to GPU T4 x2 and run again."); pending = []
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and n not in failed and (ONLY is None or n in ONLY): print(f"   {n:20s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


runs/dapt_llama/best_model.pt: reconstructing file:   0%|          |  0.00B /  431MB            

runs/dapt_llama/best_model.pt: downloading bytes:           |  0.00B            

runs/explore_shared/final_model.pt: reconstructing file:   0%|          |  0.00B /  235MB            

runs/explore_shared/final_model.pt: downloading bytes:           |  0.00B            

Phase-1 checkpoints ready
[10:14:48][_smoke] backbone explore_shared (shared) | method lorma | windows 512 tokens, 2 per case | params 60.4M trainable 1.647M
[10:14:55][_smoke] train 6 queries with relevant cases in the pool | 3 steps/epoch | lr 0.0003 | budget 35 min
[10:15:00][_smoke] before training: DEV Micro-F1@7 (dense only, 2 windows) mean 0.0364 | max 0.0545
[10:15:04][_smoke] epoch 1 (step 3): DEV Micro-F1@7 dense only - mean 0.0364 | max 0.0545 | 0.1 min
[10:15:04][_smoke] training done: 3 steps, 1 epochs, 0.1 min | best quick DEV F1 0.0545 (epoch 0)
[10:15:08][_smoke] debug run (--limit): skipping fusion
safety test passed in 30s | lexical only (BM25 words+pairs, year, hub, feedback; no model): DEV Micro-F1@7: 0.4591439689 | Micro-Precision@7: 0.5000000000 | Micro-Recall@7: 0.4244604317 | weights {'bm25b': 1.0, 'year': -2.0, 'hub': -0.5}

--- 5 min | done [] | failed []
   p7_llama_lorma       [10:20:11][p7_llama_lorma] epoch 3 step 250/2040 loss 1.4203 | 4.4 min
   p7_llama

In [9]:
# ===================== P7 CELL 8 - Results (OFFICIAL evaluator on DEV), ensembles, analysis, WHAT TO SUBMIT =====================
import numpy as np
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS:
    p = f"{OUT}/{n}/summary.json"
    if os.path.exists(p):
        s = json.load(open(p)); rows.append(s)
        for src, dst in [("submission.zip", f"{n}.zip"), ("submission_dense.zip", f"{n}_denseonly.zip")]:
            if os.path.exists(f"{OUT}/{n}/{src}"): shutil.copy(f"{OUT}/{n}/{src}", f"{ZIPS}/{dst}")
rows.sort(key=lambda s: -(s["dev_f1"] or 0))
print(f"DEV = {stats['dev']['queries']} queries, pool {stats['dev']['candidates']} cases | Micro-F1@7 by the OFFICIAL evaluate_phase7.py\n")
print(f"{'run':20s} {'model':6s} {'method':12s} {'trainable':>10s} {'dense-only':>10s} {'mean':>6s} {'max':>6s} {'FUSED':>7s} {'2-fold':>7s} "
      f"{'zero-shot':>9s} {'ep':>5s} {'train':>6s} {'GPU':>5s} {'s/case':>6s}  fusion weights")
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "")
    w = ", ".join(f"{k}={v:g}" for k, v in s["fusion_weights"].items() if v)
    print(f"{s['run']:20s} {s['arch']:6s} {m:12s} {s['params_trainable'] / 1e6:9.2f}M {s['dev_f1_dense'] or 0:10.4f} {s['dev_f1_dense_mean']:6.4f} "
          f"{s['dev_f1_dense_max']:6.4f} {s['dev_f1'] or 0:7.4f} {s['dev_f1_2fold']:7.4f} {s.get('zero_shot_quick_dev') or 0:9.4f} "
          f"{s['best_epoch']:>2}/{s['epochs_run']:<2} {s['train_minutes']:5.1f}m {s['peak_gpu_gb']:5.2f} {s['encode_seconds_per_case']:6.3f}  {w}")
print("\ndense-only = our retriever alone (best of mean / max pooling) | mean / max = case vector = mean of window vectors / MaxSim over"
      " windows | FUSED = with BM25 words+pairs, year rule, hub, feedback (weights chosen on dev) | 2-fold = weights chosen on one half"
      " of dev, scored on the other half (honest estimate) | zero-shot = quick dev check before training")
print(f"lexical only (no model): DEV {_f:.4f} (2-fold {_cx:.4f}) | train-chosen BM25: {stats['bm25']}")

# ---- ensembles: lexical + several of our retrievers together (weights chosen on dev) ----
good = [s["run"] for s in rows if "zeroshot" not in s["run"]]
ens_jobs = [("ens_all", good)] + ([("ens_top3", good[:3])] if len(good) > 3 else [])
for name, members in ens_jobs:
    if len(members) < 2: continue
    r = subprocess.run([sys.executable, f"{WORK}/p7fuse.py", "--data_dir", DATA, "--out_dir", OUT, "--runs", ",".join(members), "--name", name,
                        "--hf_repo", CKPT_REPO], capture_output=True, text=True)
    print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l))
    if os.path.exists(f"{OUT}/{name}/submission.zip"):
        r2 = subprocess.run([sys.executable, f"{WORK}/evaluate_phase7.py", "--truth", f"{DATA}/labels_dev.json", "--prediction",
                             f"{OUT}/{name}/dev_prediction.json", "--candidate-ids", f"{DATA}/dev_candidate_ids.json"], capture_output=True, text=True)
        print(f"   {name} official check:", " | ".join(l for l in r2.stdout.splitlines() if l.startswith(("Micro", "TP"))))
        shutil.copy(f"{OUT}/{name}/submission.zip", f"{ZIPS}/{name}.zip")
        s = json.load(open(f"{OUT}/{name}/summary.json")); s.update(arch="mixed", method="ensemble"); rows.append(s)

# ---- test files: every zip must be valid for the TEST pool ----
test_c = set(json.load(open(f"{DATA}/test_candidate_ids.json"))); test_q = set(str(x) for x in np.load(f"{DATA}/test_tok.npz")["q_ids"])
for z in sorted(os.listdir(ZIPS)):
    import zipfile as _zf
    p = json.loads(_zf.ZipFile(f"{ZIPS}/{z}").read("prediction.json"))
    assert set(p) == test_q and all(len(v) == 7 and len(set(v)) == 7 and set(v) <= test_c for v in p.values()), f"{z} is not valid"
print(f"\nall {len(os.listdir(ZIPS))} zips checked against the test pool (7 unique ids from test_candidates for every test query)")

# ---- submission order: one zip per technique, best LAST ----
by = {}
for s in rows:
    key = "KD" if s.get("kd_teacher") else s["method"]
    if key in ("none", "full") and s["arch"] == "shared": continue
    if key not in by or (s["dev_f1"] or 0) > (by[key]["dev_f1"] or 0): by[key] = s
plan = [by[k] for k in ["ia3", "partial", "full", "KD", "lorma"] if k in by]
best = max(rows, key=lambda s: s["dev_f1"] or 0)
if best not in plan: plan.append(best)
else: plan.sort(key=lambda s: s is best)
print("\nSUBMISSION ORDER (one zip per technique - LoRMA and KD included - best zip LAST):")
for i, s in enumerate(plan, 1):
    print(f"   {i}. zips/{s['run']}.zip  ({s['method']}{'+KD' if s.get('kd_teacher') else ''})  DEV Micro-F1@7 {s['dev_f1']:.4f}")
json.dump(rows, open(f"{P7}/p7_results.json", "w"), indent=1, default=str)
api.upload_file(path_or_fileobj=f"{P7}/p7_results.json", path_in_repo="phase7/results/p7_results.json", repo_id=CKPT_REPO, repo_type="model")
print("zips:", sorted(os.listdir(ZIPS)))


DEV = 118 queries, pool 1023 cases | Micro-F1@7 by the OFFICIAL evaluate_phase7.py

run                  model  method        trainable dense-only   mean    max   FUSED  2-fold zero-shot    ep  train   GPU s/case  fusion weights
p7_llama_lorma       llama  lorma             2.85M     0.3747 0.3580 0.3747  0.4647  0.4647    0.2368 18/19  35.5m  8.03  0.088  bm25u=0.1, bm25b=0.5, dense_max=0.5, year=-1, hub=-0.5
p7_llama_full        llama  full            107.76M     0.3635 0.3580 0.3635  0.4647  0.4548    0.2368 11/20  33.4m  8.28  0.065  bm25u=0.25, bm25b=0.5, dense_mean=0.1, dense_max=0.1, year=-1, hub=-0.5
p7_llama_partial     llama  partial          27.53M     0.3691 0.3569 0.3691  0.4647  0.4525    0.2368 18/20  22.3m  3.11  0.069  bm25u=0.25, bm25b=0.5, dense_max=0.1, year=-1, hub=-0.5
p7_shared_kd         shared full+KD          58.73M     0.2802 0.2802 0.2668  0.4636  0.4547    0.0978 12/16  35.6m  8.50  0.073  bm25u=0.25, bm25b=1.5, dense_mean=1.5, year=-2, hub=-1
p7_shared_ful